In [5]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from scipy.stats import pearsonr, spearmanr
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# 1 Load CLEAR training and validation data


BASE_DIR = Path.cwd()

DATA_DIR = Path(
    "/Users/nataliakolbina/Documents/workspace/haptic/"
    "CLEAR_original_grouped_split_2026-09-25"
)

TRAIN_PATH = (
    DATA_DIR / "CLEAR_original_train.csv"
)

VAL_PATH = (
    DATA_DIR / "CLEAR_original_validation.csv"
)

print(
    "Working directory:",
    BASE_DIR
)

print(
    "Training file:",
    TRAIN_PATH
)

print(
    "Validation file:",
    VAL_PATH
)

print(
    "Training file exists:",
    TRAIN_PATH.exists()
)

print(
    "Validation file exists:",
    VAL_PATH.exists()
)


# Load the new CLEAR training and validation splits
train = pd.read_csv(
    TRAIN_PATH
)

val = pd.read_csv(
    VAL_PATH
)


# Keep passages with valid text and BT.Easiness values
train = train.dropna(
    subset=[
        "Excerpt",
        "BT.Easiness"
    ]
).copy()

val = val.dropna(
    subset=[
        "Excerpt",
        "BT.Easiness"
    ]
).copy()


# Set the expected data types
train["Excerpt"] = (
    train["Excerpt"]
    .astype(str)
)

val["Excerpt"] = (
    val["Excerpt"]
    .astype(str)
)

train["BT.Easiness"] = (
    train["BT.Easiness"]
    .astype(float)
)

val["BT.Easiness"] = (
    val["BT.Easiness"]
    .astype(float)
)


# Confirm the final dataset sizes
print(
    "\nTrain:",
    train.shape
)

print(
    "Validation:",
    val.shape
)

print(
    "\nTraining sample:"
)

print(
    train[
        [
            "Excerpt",
            "BT.Easiness"
        ]
    ].head()
)



# 2 Load the pretrained ReadaBERT tokenizer


MODEL_NAME = "tareknaous/readabert-en"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(
    "\nTokenizer:",
    tokenizer.name_or_path
)


# 3 Check training-passage token lengths


token_lengths = []

for text in train["Excerpt"]:

    encoded = tokenizer(
        text,
        truncation=False
    )

    token_lengths.append(
        len(
            encoded["input_ids"]
        )
    )

lengths = pd.Series(
    token_lengths
)

print(
    "\nTraining token lengths:"
)

print(
    lengths.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

MAX_LENGTH = 288

print(
    "\nMaximum sequence length used:",
    MAX_LENGTH
)

print(
    "Passages longer than MAX_LENGTH:",
    int(
        (
            lengths
            > MAX_LENGTH
        ).sum()
    )
)

print(
    "Percentage longer than MAX_LENGTH:",
    round(
        (
            lengths
            > MAX_LENGTH
        ).mean()
        * 100,
        2
    )
)



# 4 Prepare CLEAR datasets for PyTorch


class CLEARDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length
    ):
        self.texts = (
            texts.tolist()
        )

        self.labels = (
            labels.tolist()
        )

        self.tokenizer = (
            tokenizer
        )

        self.max_length = (
            max_length
        )

    def __len__(self):

        return len(
            self.texts
        )

    def __getitem__(
        self,
        idx
    ):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding[
                    "input_ids"
                ].squeeze(0),

            "attention_mask":
                encoding[
                    "attention_mask"
                ].squeeze(0),

            "labels":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.float
                )
        }


train_dataset = CLEARDataset(
    train["Excerpt"],
    train["BT.Easiness"],
    tokenizer,
    MAX_LENGTH
)

val_dataset = CLEARDataset(
    val["Excerpt"],
    val["BT.Easiness"],
    tokenizer,
    MAX_LENGTH
)


train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

print(
    "\nTraining batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)



# 5 Select the available computing device


if torch.cuda.is_available():

    device = torch.device(
        "cuda"
    )

elif (
    hasattr(
        torch.backends,
        "mps"
    )
    and
    torch.backends.mps.is_available()
):

    device = torch.device(
        "mps"
    )

else:

    device = torch.device(
        "cpu"
    )

print(
    "\nDevice:",
    device
)



# 6 Define shared training and evaluation functions


def train_one_epoch(
    model,
    loader,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0

    for batch in loader:

        optimizer.zero_grad()

        input_ids = (
            batch["input_ids"]
            .to(device)
        )

        attention_mask = (
            batch["attention_mask"]
            .to(device)
        )

        labels = (
            batch["labels"]
            .to(device)
        )

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item()
        )

    return (
        total_loss
        / len(loader)
    )


def evaluate_model(
    model,
    loader,
    device
):
    model.eval()

    predictions = []
    targets = []

    with torch.no_grad():

        for batch in loader:

            input_ids = (
                batch["input_ids"]
                .to(device)
            )

            attention_mask = (
                batch[
                    "attention_mask"
                ]
                .to(device)
            )

            labels = (
                batch["labels"]
                .to(device)
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = (
                outputs.logits
                .squeeze(-1)
            )

            predictions.extend(
                preds
                .cpu()
                .numpy()
            )

            targets.extend(
                labels
                .cpu()
                .numpy()
            )

    return (
        np.asarray(
            targets
        ),
        np.asarray(
            predictions
        )
    )


def calculate_metrics(
    y_true,
    y_pred
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mae = np.mean(
        np.abs(
            y_true
            - y_pred
        )
    )

    rmse = np.sqrt(
        np.mean(
            (
                y_true
                - y_pred
            ) ** 2
        )
    )

    ss_res = np.sum(
        (
            y_true
            - y_pred
        ) ** 2
    )

    ss_tot = np.sum(
        (
            y_true
            - np.mean(y_true)
        ) ** 2
    )

    r2 = (
        1
        - (
            ss_res
            / ss_tot
        )
    )

    pearson = pearsonr(
        y_true,
        y_pred
    ).statistic

    spearman = spearmanr(
        y_true,
        y_pred
    ).statistic

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Pearson": pearson,
        "Spearman": spearman
    }



# 7 Stage 1: train regression head with backbone frozen


model_frozen = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        attn_implementation="eager",
        num_labels=1,
        ignore_mismatched_sizes=True
    )
)

model_frozen.config.problem_type = (
    "regression"
)

# Freeze the pretrained transformer backbone
for param in (
    model_frozen
    .base_model
    .parameters()
):
    param.requires_grad = False

model_frozen.to(
    device
)

trainable = sum(
    parameter.numel()
    for parameter
    in model_frozen.parameters()
    if parameter.requires_grad
)

total = sum(
    parameter.numel()
    for parameter
    in model_frozen.parameters()
)

print(
    "\nFrozen-backbone stage"
)

print(
    "Trainable parameters:",
    trainable
)

print(
    "Total parameters:",
    total
)

optimizer = AdamW(
    filter(
        lambda parameter:
            parameter.requires_grad,
        model_frozen.parameters()
    ),
    lr=1e-3
)

FROZEN_EPOCHS = 5

best_rmse = float(
    "inf"
)

FROZEN_SAVE_DIR = (
    BASE_DIR
    / "readabert_clear_frozen_best_rerun"
)

for epoch in range(
    FROZEN_EPOCHS
):

    train_loss = train_one_epoch(
        model_frozen,
        train_loader,
        optimizer,
        device
    )

    y_val, y_pred = (
        evaluate_model(
            model_frozen,
            val_loader,
            device
        )
    )

    metrics = calculate_metrics(
        y_val,
        y_pred
    )

    print(
        f"\nEpoch "
        f"{epoch + 1}/"
        f"{FROZEN_EPOCHS}"
    )

    print(
        f"Train loss: "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation MAE: "
        f"{metrics['MAE']:.4f}"
    )

    print(
        f"Validation RMSE: "
        f"{metrics['RMSE']:.4f}"
    )

    print(
        f"Validation R²: "
        f"{metrics['R2']:.4f}"
    )

    print(
        f"Validation Pearson: "
        f"{metrics['Pearson']:.4f}"
    )

    print(
        f"Validation Spearman: "
        f"{metrics['Spearman']:.4f}"
    )

    if (
        metrics["RMSE"]
        < best_rmse
    ):
        best_rmse = (
            metrics["RMSE"]
        )

        model_frozen.save_pretrained(
            FROZEN_SAVE_DIR
        )

        tokenizer.save_pretrained(
            FROZEN_SAVE_DIR
        )

        print(
            "Best frozen model saved."
        )



# 8 Evaluate the best frozen-backbone checkpoint


best_frozen_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        FROZEN_SAVE_DIR
    )
)

best_frozen_model.to(
    device
)

best_frozen_model.eval()

(
    y_val_frozen,
    y_pred_frozen
) = evaluate_model(
    best_frozen_model,
    val_loader,
    device
)

frozen_metrics = (
    calculate_metrics(
        y_val_frozen,
        y_pred_frozen
    )
)

print(
    "\nFrozen ReadaBERT "
    "— Validation Results"
)

print(
    "-------------------------------------"
)

print(
    f"MAE:      "
    f"{frozen_metrics['MAE']:.4f}"
)

print(
    f"RMSE:     "
    f"{frozen_metrics['RMSE']:.4f}"
)

print(
    f"R²:       "
    f"{frozen_metrics['R2']:.4f}"
)

print(
    f"Pearson:  "
    f"{frozen_metrics['Pearson']:.4f}"
)

print(
    f"Spearman: "
    f"{frozen_metrics['Spearman']:.4f}"
)



# 9 Plot frozen-model validation predictions


FROZEN_PLOT = (
    BASE_DIR
    / "readabert_frozen_validation_rerun.png"
)

plt.figure(
    figsize=(7, 6)
)

plt.scatter(
    y_val_frozen,
    y_pred_frozen,
    alpha=0.5
)

minimum = min(
    y_val_frozen.min(),
    y_pred_frozen.min()
)

maximum = max(
    y_val_frozen.max(),
    y_pred_frozen.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum]
)

plt.xlabel(
    "Actual BT.Easiness"
)

plt.ylabel(
    "Predicted BT.Easiness"
)

plt.title(
    "Frozen ReadaBERT: "
    "Actual vs Predicted BT.Easiness"
)

plt.tight_layout()

plt.savefig(
    FROZEN_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 10 Stage 2: fully fine-tune ReadaBERT


model_ft = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        attn_implementation="eager",
        num_labels=1,
        ignore_mismatched_sizes=True
    )
)

model_ft.config.problem_type = (
    "regression"
)

model_ft.to(
    device
)

trainable = sum(
    parameter.numel()
    for parameter
    in model_ft.parameters()
    if parameter.requires_grad
)

total = sum(
    parameter.numel()
    for parameter
    in model_ft.parameters()
)

print(
    "\nFull fine-tuning stage"
)

print(
    "Trainable parameters:",
    trainable
)

print(
    "Total parameters:",
    total
)

print(
    "All trainable:",
    trainable == total
)

optimizer_ft = AdamW(
    model_ft.parameters(),
    lr=2e-5
)

FINETUNE_EPOCHS = 1

best_rmse_ft = float(
    "inf"
)

FINETUNED_SAVE_DIR = (
    BASE_DIR
    / "readabert_clear_finetuned_best_rerun"
)

start = time.time()

for epoch in range(
    FINETUNE_EPOCHS
):

    epoch_start = (
        time.time()
    )

    train_loss = train_one_epoch(
        model_ft,
        train_loader,
        optimizer_ft,
        device
    )

    y_val, y_pred = (
        evaluate_model(
            model_ft,
            val_loader,
            device
        )
    )

    metrics = calculate_metrics(
        y_val,
        y_pred
    )

    print(
        f"\nEpoch "
        f"{epoch + 1}/"
        f"{FINETUNE_EPOCHS}"
    )

    print(
        f"Train loss: "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation MAE: "
        f"{metrics['MAE']:.4f}"
    )

    print(
        f"Validation RMSE: "
        f"{metrics['RMSE']:.4f}"
    )

    print(
        f"Validation R²: "
        f"{metrics['R2']:.4f}"
    )

    print(
        f"Validation Pearson: "
        f"{metrics['Pearson']:.4f}"
    )

    print(
        f"Validation Spearman: "
        f"{metrics['Spearman']:.4f}"
    )

    print(
        "Epoch time (min): "
        f"{(time.time() - epoch_start) / 60:.2f}"
    )

    print(
        "Total time (min): "
        f"{(time.time() - start) / 60:.2f}"
    )

    if (
        metrics["RMSE"]
        < best_rmse_ft
    ):
        best_rmse_ft = (
            metrics["RMSE"]
        )

        model_ft.save_pretrained(
            FINETUNED_SAVE_DIR
        )

        tokenizer.save_pretrained(
            FINETUNED_SAVE_DIR
        )

        print(
            "Best fine-tuned model saved."
        )



# 11 Evaluate the best fully fine-tuned checkpoint


model_ft_best = (
    AutoModelForSequenceClassification
    .from_pretrained(
        FINETUNED_SAVE_DIR
    )
)

model_ft_best.to(
    device
)

model_ft_best.eval()

(
    y_val_ft,
    y_pred_ft
) = evaluate_model(
    model_ft_best,
    val_loader,
    device
)

metrics_ft = calculate_metrics(
    y_val_ft,
    y_pred_ft
)

print(
    "\nFine-tuned ReadaBERT "
    "— Validation Results"
)

print(
    "------------------------------------------"
)

print(
    f"MAE:      "
    f"{metrics_ft['MAE']:.4f}"
)

print(
    f"RMSE:     "
    f"{metrics_ft['RMSE']:.4f}"
)

print(
    f"R²:       "
    f"{metrics_ft['R2']:.4f}"
)

print(
    f"Pearson:  "
    f"{metrics_ft['Pearson']:.4f}"
)

print(
    f"Spearman: "
    f"{metrics_ft['Spearman']:.4f}"
)


# 12 Plot fully fine-tuned validation predictions


FINETUNED_PLOT = (
    BASE_DIR
    / "readabert_finetuned_validation_rerun.png"
)

plt.figure(
    figsize=(7, 6)
)

plt.scatter(
    y_val_ft,
    y_pred_ft,
    alpha=0.5
)

minimum = min(
    y_val_ft.min(),
    y_pred_ft.min()
)

maximum = max(
    y_val_ft.max(),
    y_pred_ft.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum]
)

plt.xlabel(
    "Actual BT.Easiness"
)

plt.ylabel(
    "Predicted BT.Easiness"
)

plt.title(
    "Fine-tuned ReadaBERT: "
    "Actual vs Predicted BT.Easiness"
)

plt.tight_layout()

plt.savefig(
    FINETUNED_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 13 Compare frozen and fully fine-tuned validation results


comparison_df = pd.DataFrame(
    [
        {
            "Model": "Frozen backbone",
            "MAE": frozen_metrics["MAE"],
            "RMSE": frozen_metrics["RMSE"],
            "R2": frozen_metrics["R2"],
            "Pearson": frozen_metrics["Pearson"],
            "Spearman": frozen_metrics["Spearman"]
        },
        {
            "Model": "Fully fine-tuned",
            "MAE": metrics_ft["MAE"],
            "RMSE": metrics_ft["RMSE"],
            "R2": metrics_ft["R2"],
            "Pearson": metrics_ft["Pearson"],
            "Spearman": metrics_ft["Spearman"]
        }
    ]
)

print(
    "\nVALIDATION MODEL COMPARISON"
)

print(
    comparison_df.round(4)
)



# 14 Save validation comparison results


COMPARISON_PATH = (
    BASE_DIR
    / "phase5_validation_model_comparison_rerun.csv"
)

comparison_df.to_csv(
    COMPARISON_PATH,
    index=False
)

print(
    "\nSaved:",
    COMPARISON_PATH
)



# 15 Confirm Phase 5 outputs


print(
    "\nPHASE 5 COMPLETE"
)

print(
    "Frozen checkpoint:",
    FROZEN_SAVE_DIR
)

print(
    "Fine-tuned checkpoint:",
    FINETUNED_SAVE_DIR
)

print(
    "Frozen validation plot:",
    FROZEN_PLOT
)

print(
    "Fine-tuned validation plot:",
    FINETUNED_PLOT
)

print(
    "Validation comparison:",
    COMPARISON_PATH
)

Working directory: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++
Training file: /Users/nataliakolbina/Documents/workspace/haptic/CLEAR_original_grouped_split_2026-09-25/CLEAR_original_train.csv
Validation file: /Users/nataliakolbina/Documents/workspace/haptic/CLEAR_original_grouped_split_2026-09-25/CLEAR_original_validation.csv
Training file exists: True
Validation file exists: True

Train: (3307, 14)
Validation: (708, 14)

Training sample:
                                             Excerpt  BT.Easiness
0  When the young people returned to the ballroom...    -0.340259
1  All through dinner time, Mrs. Fayre was somewh...    -0.315372
2  As Roger had predicted, the snow departed as q...    -0.580118
3  Mr. Grimes was to come up next morning to Sir ...    -1.785965
4  Hal and Chester found ample time to take an in...    -0.861809

Tokenizer: tareknaous/readabert-en

Training token lengths:
count    3307.000000
mean      217.853644
std        25.591013
min     

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at tareknaous/readabert-en and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([6]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([6, 768]) in the checkpoint and torch.Size([1, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Frozen-backbone stage
Trainable parameters: 769
Total parameters: 109483009

Epoch 1/5
Train loss: 0.7384
Validation MAE: 0.6124
Validation RMSE: 0.7684
Validation R²: 0.4600
Validation Pearson: 0.6903
Validation Spearman: 0.7046
Best frozen model saved.

Epoch 2/5
Train loss: 0.6678
Validation MAE: 0.6753
Validation RMSE: 0.8409
Validation R²: 0.3534
Validation Pearson: 0.7014
Validation Spearman: 0.7122

Epoch 3/5
Train loss: 0.6461
Validation MAE: 0.5848
Validation RMSE: 0.7377
Validation R²: 0.5023
Validation Pearson: 0.7184
Validation Spearman: 0.7343
Best frozen model saved.

Epoch 4/5
Train loss: 0.6170
Validation MAE: 0.7349
Validation RMSE: 0.9018
Validation R²: 0.2564
Validation Pearson: 0.7216
Validation Spearman: 0.7367

Epoch 5/5
Train loss: 0.6294
Validation MAE: 0.7501
Validation RMSE: 0.9186
Validation R²: 0.2283
Validation Pearson: 0.7365
Validation Spearman: 0.7519

Frozen ReadaBERT — Validation Results
-------------------------------------
MAE:      0.5848
RMSE:    

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at tareknaous/readabert-en and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([6]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([6, 768]) in the checkpoint and torch.Size([1, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Full fine-tuning stage
Trainable parameters: 109483009
Total parameters: 109483009
All trainable: True

Epoch 1/1
Train loss: 0.4218
Validation MAE: 0.4038
Validation RMSE: 0.5120
Validation R²: 0.7603
Validation Pearson: 0.8750
Validation Spearman: 0.8878
Epoch time (min): 4.31
Total time (min): 4.31
Best fine-tuned model saved.

Fine-tuned ReadaBERT — Validation Results
------------------------------------------
MAE:      0.4038
RMSE:     0.5120
R²:       0.7603
Pearson:  0.8750
Spearman: 0.8878

VALIDATION MODEL COMPARISON
              Model     MAE    RMSE      R2  Pearson  Spearman
0   Frozen backbone  0.5848  0.7377  0.5023   0.7184    0.7343
1  Fully fine-tuned  0.4038  0.5120  0.7603   0.8750    0.8878

Saved: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/phase5_validation_model_comparison_rerun.csv

PHASE 5 COMPLETE
Frozen checkpoint: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/readabert_clear_frozen_best_rerun
Fine-tun